[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/exorbyte/mbox-cookbook/blob/main/05-explainability/05-ident_fields_explained.ipynb)

In [1]:
# !pip install mbox

# IDENT Fields Explained

`IndexType.IDENT` is for identifiers: SKUs, product codes, tax IDs, passport numbers. It shares its comparison engine with `TERM`, both work character by character across the whole string, not word by word like `PHRASE`. `01-index_types_and_recall_modes.ipynb` showed `IDENT` cutting off one edit sooner than `TERM` on the word `"APPROXIMATELY"`, and called that difference deliberate. This notebook tests that claim properly, on data `IDENT` is actually meant for, and finds something more nuanced than "always stricter."

In this notebook you will:

1. Confirm `APPROX` on `IDENT` still gives typo-tolerant, graded scores on real product codes
2. Test whether `IDENT`'s cutoff is actually stricter than `TERM`'s on a realistic identifier, and find a case where it is not
3. Discover that common separator characters, dashes, spaces, underscores, are treated as equivalent, but removing a separator entirely is not
4. Confirm `COMPLETE` and `EXACT` behave exactly like they do on `TERM`
5. Walk away with practical guidance for identifier fields, including when to stop trusting fuzziness altogether

In [2]:
import pandas as pd
from mbox.indexing import TableIndexer
from mbox.recall import TableRecallMode

catalog = pd.read_csv("datasets/product_catalog.csv")
index = TableIndexer.create_index(
    catalog,
    index_columns=["product_id", "product_name", "category", "year_released", "price"],
    tmp_dir="tmp_index"
)

catalog[["product_id", "product_name"]]

mbpie: 33 modules, 566 methods, 8 classes, 18 enums
  args: 426 required, 254 optional, 37 keywords, 39 flags, 26 arrays
  types: 372 int, 337 str, 1 double, 72 object


,product_id,product_name
0,B88-EXT-24,Extended Battery Pack Pro
1,A12-PWR-22,Portable Power Bank
2,C99-SNS-23,Motion Sensor Camera
3,D45-REL-21,Smart Relay Switch
4,E67-LEN-24,Wide Angle Camera Lens
5,F31-TRK-20,GPS Tracker Module
6,G14-CAB-19,Braided USB-C Cable
7,H82-DOC-23,Docking Station Hub
8,I29-BAT-22,Rechargeable Battery Cell
9,J56-SEN-24,Outdoor Motion Sensor


`product_id` is inferred as `IDENT`: short, structured codes like `"B88-EXT-24"`, mixing letters, digits, and dashes.

## 1. `APPROX` on `IDENT` is still typo-tolerant

A code with one or two characters off still resolves, with a graded score, exactly the same shape of behavior you have seen on `PHRASE` and `TERM`.

In [3]:
id_queries = ["B88-EXT-24", "B88-EXY-24", "B89-EXT-25", "888-EXT-24"]

rows = []
for query in id_queries:
    result = index.match(product_id=query, modes={"product_id": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"query": query, "found": found, "product_id_score": result["product_id_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,query,found,product_id_score
0,B88-EXT-24,True,100
1,B88-EXY-24,True,81
2,B89-EXT-25,True,61
3,888-EXT-24,True,81


## 2. Is `IDENT` actually stricter than `TERM`? It depends on the string

`01-index_types_and_recall_modes.ipynb` found `IDENT` dropping a match one edit sooner than `TERM` on `"APPROXIMATELY"`. Let's run the identical kind of test on `"B88-EXT-24"`, a realistic 10-character identifier, this time comparing `IDENT` directly against a `TERM`-typed copy of the same value.

In [4]:
from mbox.config import TableConfig, TableFieldConfig, IndexType

def substitute_n(s, n, seed=0):
    """Return a copy of s with exactly n characters substituted, at fixed positions."""
    import random
    random.seed(seed)
    s = list(s)
    positions = random.sample(range(len(s)), n)
    for p in positions:
        s[p] = "Q" if s[p] != "Q" else "Z"
    return "".join(s)

def build_single_column_index(index_type):
    df = pd.DataFrame({"code": ["B88-EXT-24"]})
    config = TableConfig(fields=[TableFieldConfig(column="code", index_type=index_type)])
    return TableIndexer.create_index(df=df, config_overrides=config, tmp_dir="tmp_index")

rows = []
for index_type in [IndexType.IDENT, IndexType.TERM]:
    code_index = build_single_column_index(index_type)
    for n_edits in range(0, 5):
        query = substitute_n("B88-EXT-24", n_edits) if n_edits else "B88-EXT-24"
        result = code_index.match(code=query, modes={"code": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0)
        found = len(result) > 0 and result["index_row"].iloc[0] != -1
        rows.append({"index_type": index_type.name, "edits": n_edits, "code_score": result["code_score"].iloc[0] if found else None})

pd.DataFrame(rows).pivot(index="edits", columns="index_type", values="code_score")

config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.
config_overrides is given, therefore all information from index_columns, index_types, alias_sets and character_mappings is ignored.


index_type,IDENT,TERM
edits,,
0,100.0,100.0
1,81.0,81.0
2,61.0,61.0
3,40.0,40.0
4,NaN,NaN


On this particular string, `IDENT` and `TERM` cut off at exactly the same point. The stricter cutoff you saw in the overview notebook is real, but it is not a fixed "one edit fewer" rule you can count on for every string, it depends on the specific characters and length involved. The one thing that does hold: `IDENT`'s cutoff is never *more* generous than `TERM`'s, only equal or stricter.

The practical consequence is a caution, not a comfort. If a false positive on an identifier is genuinely costly, a tax ID resolving to the wrong taxpayer, a SKU resolving to the wrong part, do not rely on `IDENT`'s cutoff alone to catch it. Set an explicit `minimum_quality` floor with `TableRecallConfig`, covered in `04-recall-tuning/02-weighting_fields_for_better_precision.ipynb`. `IDENT` is a sensible default for identifiers; it is not, by itself, a guarantee.

## 3. Separators: dashes, spaces, and underscores are equivalent, missing is not

Real identifiers rarely arrive formatted consistently across systems, one export uses dashes, another uses spaces, a third strips punctuation entirely. Here is how `APPROX` on `IDENT` handles each variation of `"B88-EXT-24"`.

In [5]:
separator_queries = [
    ("original", "B88-EXT-24"),
    ("dash replaced with space", "B88 EXT 24"),
    ("dash replaced with underscore", "B88_EXT_24"),
    ("lowercase", "b88-ext-24"),
    ("separators removed entirely", "B88EXT24"),
]

rows = []
for label, query in separator_queries:
    result = index.match(product_id=query, modes={"product_id": TableRecallMode.APPROX}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    rows.append({"scenario": label, "query": query, "found": found, "product_id_score": result["product_id_score"].iloc[0] if found else None})

pd.DataFrame(rows)

,scenario,query,found,product_id_score
0,original,B88-EXT-24,True,100
1,dash replaced with space,B88 EXT 24,True,100
2,dash replaced with underscore,B88_EXT_24,True,100
3,lowercase,b88-ext-24,True,100
4,separators removed entirely,B88EXT24,True,61


Swapping the dash for a space or an underscore costs nothing, both score `100`, right alongside the lowercase variant. Removing the separators entirely is different: `"B88EXT24"` is two characters shorter than the indexed value, and `APPROX` charges for those two missing positions the same way it would charge for any other deletion. If your identifiers arrive with inconsistent punctuation across source systems, this is usually forgiven automatically. If they arrive with punctuation sometimes present and sometimes stripped out completely, it is not, and you may want to normalize that away explicitly with `CharacterMapping` before indexing, covered in `02-data-harmonization/`, rather than relying on `APPROX` to absorb it.

## 4. `COMPLETE` and `EXACT`: identical mechanics to `TERM`

`IDENT` and `TERM` share the same character-sequence engine, so the anchoring and normalization rules from `04-term_fields_explained.ipynb` carry over unchanged. `COMPLETE` only finds fragments anchored at the start of the value, and `EXACT` normalizes case but not whitespace.

In [6]:
print("COMPLETE, prefix vs. unanchored fragment:")
for query in ["B88-EXT", "EXT-24"]:
    result = index.match(product_id=query, modes={"product_id": TableRecallMode.COMPLETE}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    print(f"  {query!r:12s} found={found}")

print("\nEXACT, case vs. surrounding whitespace:")
for query in ["b88-ext-24", " B88-EXT-24 "]:
    result = index.match(product_id=query, modes={"product_id": TableRecallMode.EXACT}, include_field_scores=True, min_total_match_value=0, max_results=1)
    found = len(result) > 0 and result["index_row"].iloc[0] != -1
    print(f"  {query!r:15s} found={found}")

COMPLETE, prefix vs. unanchored fragment:
  'B88-EXT'    found=True
  'EXT-24'     found=False

EXACT, case vs. surrounding whitespace:
  'b88-ext-24'    found=True
  ' B88-EXT-24 '  found=False


The prefix `"B88-EXT"` is found under `COMPLETE`, the unanchored fragment `"EXT-24"` is not, same rule as `TERM`. `EXACT` tolerates the case change and rejects the surrounding whitespace, also identical to `TERM`.

## 5. Practical guidance for `IDENT` fields

**Use `IDENT` for anything you would call a code or an identifier**, SKUs, tax IDs, passport numbers, transaction references. It signals intent even where its cutoff happens to coincide with `TERM`'s, and on many real identifiers it will be measurably stricter.

**Do not treat `IDENT`'s cutoff as a guaranteed safety margin.** Section 2 showed a realistic code where `IDENT` and `TERM` disagreed on nothing. If a false positive on this field is costly, back it with an explicit `minimum_quality`, don't rely on `APPROX`'s internal cutoff to catch what a business rule should.

**Separator inconsistency is usually free, disappearance is not.** Dashes, spaces, and underscores interchange with no score penalty. If some of your source data drops separators entirely, decide up front whether to normalize that with `CharacterMapping` or simply accept the lower score, letting it happen silently is how a "why didn't this match" ticket gets filed later.

**For identifiers where even a small amount of tolerance is unacceptable, don't lean on `IDENT`'s fuzziness at all, use `EXACT`.** A one-character-off tax ID or transaction reference is not "close enough," it's wrong, and `04-recall-tuning/01-understanding_match_modes.ipynb` already covers exactly when to reach for `EXACT` instead of `APPROX`.

## Next steps

<!-- - **`06-numeric_fields_explained.ipynb`** - the same kind of investigation, for `INTEGER` and `DOUBLE` fields
- **`07-multi_field_weights_explained.ipynb`** - combining an `IDENT` field with others into one `overall_score` -->
- **`06-debugging_a_bad_match.ipynb`** - a real, multi-cause debugging walkthrough that draws on everything in this directory

*M|BOX is currently in `beta`. Breaking changes may occur in minor releases until version `1.0.0`.*